# 🔬 Notebook 3: Deep Dive — BFS, PYMK, and Hot Users

This notebook is where we earn our keep. We'll take three real problems and show **bad → best**:

1. **Degrees of separation** — one-sided BFS vs *bidirectional* BFS.
2. **People You May Know** — naive live scoring vs precomputed offline pipeline.
3. **The celebrity problem** — what happens when one user has 30k neighbors, and how to survive it.

All code is runnable on a small graph you can read. Then we generate a bigger power-law graph and watch the bad solutions collapse.

## 🛠️ Setup

```bash
cd 06-system-designs/linkedin-connections
uv sync
```

Then select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab has **no external services** (no Postgres, no Redis). Everything runs in-process with Python stdlib + `pydantic`, so you can focus on the concepts.

## 0. A reusable toy graph

We represent the graph as `dict[int, set[int]]` — an adjacency list. Real systems use the same shape; they just shard it.

In [1]:
from collections import defaultdict, deque, Counter
import random, time

def build_graph(edges):
    g = defaultdict(set)
    for u, v in edges:
        if u == v: continue
        g[u].add(v); g[v].add(u)
    return g

# A hand-made graph with an obvious structure so we can eyeball answers.
small = build_graph([
    (1,2),(2,3),(3,4),(4,5),     # chain 1-2-3-4-5
    (1,6),(6,7),(7,5),           # detour 1-6-7-5
    (3,8),(8,9),(9,10),          # branch off 3
])
for u in sorted(small):
    print(f'{u}: {sorted(small[u])}')

1: [2, 6]
2: [1, 3]
3: [2, 4, 8]
4: [3, 5]
5: [4, 7]
6: [1, 7]
7: [5, 6]
8: [3, 9]
9: [8, 10]
10: [9]


## 1. Degrees of separation

### ❌ Bad — single-sided BFS up to depth 3

A BFS from `u` explores **all** users at distance 1, then 2, then 3. If the average user has `b` neighbors, at depth `d` we visit roughly `b^d` nodes.

For `b = 500` that's **500 → 250k → 125M** nodes. Doable for depth 2, hopeless for depth 3 unless you cap aggressively.

In [2]:
def bfs_one_sided(g, src, dst, max_depth=3):
    if src == dst: return 0, 1
    seen = {src}
    frontier = deque([(src, 0)])
    visited = 0
    while frontier:
        node, d = frontier.popleft()
        visited += 1
        if d == max_depth: continue
        for nb in g[node]:
            if nb == dst: return d + 1, visited + 1
            if nb not in seen:
                seen.add(nb); frontier.append((nb, d + 1))
    return None, visited

print(bfs_one_sided(small, 1, 10))  # (degree, nodes_visited)

(None, 8)


### ✅ Best — **bidirectional** BFS

Trick: BFS from **both** ends simultaneously, alternating sides, and stop as soon as the two frontiers meet.
Instead of `b^d` work, we do roughly `2·b^(d/2)` — the square root of the bad version.

For `b=500, d=4`: bad = 62 billion, good = 500k. **Five orders of magnitude faster.**

In [3]:
def bfs_bidirectional(g, src, dst, max_depth=4):
    if src == dst: return 0, 1
    # Two frontiers + the distance we've reached from each side.
    front_s = {src: 0}
    front_t = {dst: 0}
    visited = 2
    while front_s and front_t:
        # Always expand the smaller frontier first — keeps work balanced.
        if len(front_s) > len(front_t):
            front_s, front_t = front_t, front_s
        next_front = {}
        for node, d in front_s.items():
            for nb in g[node]:
                if nb in front_t:
                    return d + 1 + front_t[nb], visited + 1
                if nb not in front_s and nb not in next_front:
                    next_front[nb] = d + 1
                    visited += 1
        # depth bound: sum of depths from both sides
        if min(front_s.values()) + min(front_t.values() or [0]) > max_depth:
            return None, visited
        front_s = next_front
    return None, visited

print(bfs_bidirectional(small, 1, 10))

(5, 12)


### Benchmark on a larger synthetic graph

Let's build a preferential-attachment graph (rich get richer) — the same shape as real social networks.

In [4]:
def preferential_attachment(n_nodes=5_000, m=5, seed=42):
    rng = random.Random(seed)
    g = defaultdict(set)
    # seed clique
    for i in range(m):
        for j in range(i+1, m):
            g[i].add(j); g[j].add(i)
    degree_bag = []
    for i in range(m):
        degree_bag.extend([i] * (m-1))
    for new in range(m, n_nodes):
        targets = set()
        while len(targets) < m:
            targets.add(rng.choice(degree_bag))
        for t in targets:
            g[new].add(t); g[t].add(new)
            degree_bag.extend([new, t])
    return g

big = preferential_attachment(n_nodes=5_000, m=6)
print('nodes:', len(big), '  edges:', sum(len(v) for v in big.values()) // 2)
print('max degree:', max(len(v) for v in big.values()), '(this is the "celebrity")')

src, dst = 0, 4999
t = time.perf_counter(); d1, n1 = bfs_one_sided(big, src, dst, max_depth=5); t1 = time.perf_counter()-t
t = time.perf_counter(); d2, n2 = bfs_bidirectional(big, src, dst, max_depth=5); t2 = time.perf_counter()-t
print(f'one-sided  : degree={d1}  visited={n1:,}  time={t1*1000:.1f} ms')
print(f'bidirect.  : degree={d2}  visited={n2:,}  time={t2*1000:.1f} ms')
print(f'speedup    : {t1/t2:.1f}x   (and {n1/max(n2,1):.1f}x fewer nodes explored)')

nodes: 5000   edges: 29979
max degree: 285 (this is the "celebrity")
one-sided  : degree=3  visited=366  time=0.9 ms
bidirect.  : degree=3  visited=160  time=0.1 ms
speedup    : 13.6x   (and 2.3x fewer nodes explored)


### Production note

LinkedIn doesn't run BFS on its main graph DB per request — the query fan-out would melt any database. In practice the service:

- caches each user's **1st-degree** neighbors in a fast KV store (hot adjacency),
- computes **2nd-degree** by unioning the 1st-degree sets of the user's friends (with de-dup),
- **caps at degree 3** because the world is small: in a graph of 1B nodes, almost everyone is within 4-5 hops.

## 2. People You May Know (PYMK)

**Intuition.** If you and Carol share 12 common friends, you probably know each other.
Formally: for each non-neighbor `c` of `u`, score `c` by the count of paths of length 2 between `u` and `c`, i.e. `|N(u) ∩ N(c)|`.

### ❌ Bad — live, unweighted, per-request

Compute candidates on every page view. For a user with 500 friends and each friend with 500 friends, we touch ~250k pairs **per request**.

In [5]:
def pymk_live_naive(g, u, top_k=5):
    direct = g[u]
    scores = Counter()
    for f in direct:
        for fof in g[f]:
            if fof == u or fof in direct: continue
            scores[fof] += 1
    return scores.most_common(top_k)

print('pymk_live(1) in small graph:', pymk_live_naive(small, 1))

t = time.perf_counter()
for _ in range(50):
    pymk_live_naive(big, 7)   # one repeat user
print(f'50 live calls on 5k-node graph: {(time.perf_counter()-t)*1000:.0f} ms')

pymk_live(1) in small graph: [(3, 1), (7, 1)]
50 live calls on 5k-node graph: 42 ms


### 🟡 Better — weight by mutual-friend quality (Adamic–Adar)

Raw mutual-friend count is misled by celebrities: if a celebrity with 30k friends is mutual with everyone, they make every suggestion look strong.

**Adamic–Adar** down-weights high-degree mutuals: a shared friend with only 5 friends carries more signal than one with 30,000.

$$\text{score}(u, c) = \sum_{m \in N(u) \cap N(c)} \frac{1}{\log |N(m)|}$$

In [6]:
import math

def pymk_adamic_adar(g, u, top_k=5):
    direct = g[u]
    scores = defaultdict(float)
    for f in direct:
        weight = 1.0 / math.log(len(g[f]) + 1e-9)  # shared friends with big degree count less
        for fof in g[f]:
            if fof == u or fof in direct: continue
            scores[fof] += weight
    return sorted(scores.items(), key=lambda kv: -kv[1])[:top_k]

print('naive    :', pymk_live_naive(big, 7, top_k=5))
print('adamic/aa:', [(n, round(s, 3)) for n, s in pymk_adamic_adar(big, 7, top_k=5)])

naive    : [(4, 21), (24, 19), (12, 18), (26, 14), (133, 13)]
adamic/aa: [(4, 5.831), (24, 5.466), (12, 4.746), (133, 3.889), (97, 3.881)]


### ✅ Best — precompute offline, serve from KV

Real PYMK runs in a **nightly batch** (Spark / Flink). For every user we store `top_K` suggestions in a KV store keyed by user id. At read time the app just does `GET pymk:{user_id}` — O(1).

We'll simulate the offline job with a dict. In reality the 'dict' is Redis or RocksDB, and the computation is MapReduce over a billion users.

In [7]:
def offline_pymk_job(g, top_k=5):
    out = {}
    for u in g:
        out[u] = pymk_adamic_adar(g, u, top_k)
    return out   # written to KV in production

pymk_store = offline_pymk_job(big, top_k=5)

# Serving: cheap lookup.
def serve_pymk(user_id):
    return pymk_store.get(user_id, [])

t = time.perf_counter()
for _ in range(50_000):
    serve_pymk(7)
print(f'50,000 serve_pymk calls: {(time.perf_counter()-t)*1000:.0f} ms')
print('top suggestion for user 7:', serve_pymk(7)[0])

50,000 serve_pymk calls: 2 ms
top suggestion for user 7: (4, 5.831425956958478)


### Why this architecture wins

| Dimension | Live naive | Offline + KV serve |
|---|---|---|
| Read latency | 10–500 ms (fan-out) | <1 ms (single KV hit) |
| Load on graph DB | 1× per page view | 0× at read time |
| Freshness | real-time | hours–day-old (fine for PYMK) |
| Cost | CPU every read | CPU once per day |

**Rule of thumb**: if a feature tolerates staleness, precompute it. If it needs to be live, cache it.

## 3. The celebrity / hot-user problem

Some users (Bill Gates, Richard Branson) have **hundreds of thousands** of connections. They create three real-world problems:

1. **Fat row**: reading their adjacency list returns megabytes of ids.
2. **Hot shard**: every query touching them hits one shard.
3. **Skewed PYMK**: every friend-of-friend path goes through them, dominating scores (that's why we used Adamic–Adar above).

Let's see the skew in our generated graph and then three mitigations.

In [8]:
degrees = sorted((len(v) for v in big.values()), reverse=True)
print('top-10 degrees (celebrities):', degrees[:10])
print('median degree             :', degrees[len(degrees)//2])
# Notice the tail — a classic power law.

top-10 degrees (celebrities): [285, 237, 229, 221, 217, 180, 154, 153, 149, 142]
median degree             : 8


### Mitigation 1 — pagination + capped fan-out

Never return more than `N` neighbors in one API call. Cursor-paginate. Enforce a server-side max (e.g. 100).

### Mitigation 2 — cache celebrity adjacency aggressively

Celebrities' lists don't change often relative to traffic. Cache them with a **long TTL** and **background refresh**; stampede-guard with a single-flight lock.

### Mitigation 3 — exclude or down-weight high-degree mutuals in PYMK

We already did this with Adamic–Adar. An alternative: hard-skip users whose degree exceeds some threshold when computing friend-of-friend scores.

In [9]:
def pymk_skip_celebs(g, u, top_k=5, degree_cap=500):
    direct = g[u]
    scores = Counter()
    for f in direct:
        if len(g[f]) > degree_cap:
            continue            # skip 'everyone-knows-everyone' bridges
        for fof in g[f]:
            if fof == u or fof in direct: continue
            scores[fof] += 1
    return scores.most_common(top_k)

print('without celeb filter:', pymk_live_naive(big, 7, top_k=5))
print('with    celeb filter:', pymk_skip_celebs(big, 7, top_k=5, degree_cap=200))

without celeb filter: [(4, 21), (24, 19), (12, 18), (26, 14), (133, 13)]
with    celeb filter: [(4, 17), (24, 17), (12, 15), (26, 12), (20, 11)]


## 4. One more real-world wrinkle: **sharding**

When edges are split across 1,000 database shards by `owner`, **no single shard can answer 'friends of friends'** — the 500 friends live on ~500 different shards.

Production systems (Facebook TAO, LinkedIn's graph DB) solve this with:

- A **routing layer** that fans out to the right shards in parallel.
- **Cache stickiness**: every 2nd-degree query checks the cache first; shards are touched only on misses.
- **Hedged requests**: after P95 latency, re-issue to a replica and take whichever wins.

The algorithms in this notebook don't change — only *where* they run. Your Python BFS becomes a distributed BFS with the same shape.

## 5. Closing thoughts

- **BFS everywhere** — it's the Swiss Army knife of social graphs. Bidirectional BFS is a 1000× win for free.
- **Precompute slow things, cache hot things.** PYMK is slow → precompute. Adjacency is hot → cache.
- **Mind the power law.** Celebrities break naive algorithms; Adamic–Adar, degree caps, and heavy caching are the three practical fixes.

You now have all the mental tools to reason about a billion-node social graph, and runnable code for every idea in this lab.